In [8]:
#!/usr/bin/env python3
"""Entrena con recortes WAV y busca posibles disparos en otro WAV."""

import argparse
import csv
import math
from pathlib import Path

import numpy as np
from scipy.io import wavfile
from scipy.signal import find_peaks, resample_poly
from sklearn.ensemble import RandomForestClassifier
from pyAudioAnalysis import ShortTermFeatures

FS = 16000
FRAME = 160  # 10 ms
BEFORE = int(0.08 * FS)
AFTER = int(0.22 * FS)


def leer_wav(ruta):
    fs, audio = wavfile.read(ruta)
    if audio.size == 0:
        raise ValueError(f"WAV vacío: {ruta}")
    if np.issubdtype(audio.dtype, np.unsignedinteger):
        punto_medio = (np.iinfo(audio.dtype).max + 1) / 2
        audio = (audio.astype(np.float64) - punto_medio) / punto_medio
    elif np.issubdtype(audio.dtype, np.signedinteger):
        audio = audio.astype(np.float64) / (np.iinfo(audio.dtype).max + 1)
    elif np.issubdtype(audio.dtype, np.floating):
        audio = audio.astype(np.float64)
    else:
        raise ValueError(f"Formato WAV no compatible: {ruta}")
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    if audio.ndim != 1 or not np.isfinite(audio).all():
        raise ValueError(f"Audio inválido: {ruta}")
    if fs != FS:
        divisor = math.gcd(fs, FS)
        audio = resample_poly(audio, FS // divisor, fs // divisor)
    return audio


def energias(audio):
    relleno = (-len(audio)) % FRAME
    cuadros = np.pad(audio, (0, relleno)).reshape(-1, FRAME)
    return np.sqrt(np.mean(cuadros ** 2, axis=1))


def recorte(audio, muestra):
    inicio = muestra - BEFORE
    fin = muestra + AFTER
    izquierda = max(0, -inicio)
    derecha = max(0, fin - len(audio))
    return np.pad(audio[max(0, inicio):min(len(audio), fin)],
                  (izquierda, derecha))


def caracteristicas(audio, muestra):
    segmento = recorte(audio, muestra)
    f, _ = ShortTermFeatures.feature_extraction(
        segmento, FS, int(0.025 * FS), int(0.010 * FS), deltas=False
    )
    vector = np.concatenate((f.mean(axis=1), f.std(axis=1)))
    if not np.isfinite(vector).all():
        raise ValueError("Se obtuvieron características no finitas.")
    return vector


def entrenar(carpeta):
    xs, ys = [], []
    for clase, etiqueta in (("no_disparo", 0), ("disparo", 1)):
        archivos = sorted((carpeta / clase).rglob("*.wav"))
        if not archivos:
            raise ValueError(f"No se encontraron WAV en {carpeta / clase}")
        for ruta in archivos:
            audio = leer_wav(ruta)
            energia = energias(audio)
            if energia.max() == 0:
                raise ValueError(f"Audio silencioso: {ruta}")
            muestra = min(int(np.argmax(energia)) * FRAME + FRAME // 2,
                          len(audio) - 1)
            xs.append(caracteristicas(audio, muestra))
            ys.append(etiqueta)
        print(f"{clase}: {len(archivos)} recortes")
    modelo = RandomForestClassifier(
        n_estimators=1000, class_weight="balanced", random_state=42, n_jobs=-1
    )
    modelo.fit(xs, ys)
    return modelo


def contar(modelo, audio, salida, umbral, prominencia, separacion):
    energia = energias(audio)
    maximo = energia.max()
    if maximo == 0:
        indices = np.array([], dtype=int)
    else:
        # El relleno permite detectar picos en el primer y último cuadro.
        indices, _ = find_peaks(
            np.pad(energia, (1, 1)),
            prominence=prominencia * maximo,
            distance=max(1, math.ceil(separacion / 0.01)),
        )
        indices = indices - 1
        indices = indices[(indices >= 0) & (indices < len(energia))]

    total = 0
    with salida.open("w", newline="", encoding="utf-8") as archivo:
        escritor = csv.writer(archivo)
        escritor.writerow(["tiempo_segundos", "puntaje", "clasificacion"])
        for indice in indices:
            muestra = min(int(indice) * FRAME + FRAME // 2, len(audio) - 1)
            x = caracteristicas(audio, muestra)
            puntaje = modelo.predict_proba([x])[0, 1]
            disparo = puntaje >= umbral
            total += int(disparo)
            escritor.writerow([f"{muestra / FS:.3f}", f"{puntaje:.4f}",
                                "posible_disparo" if disparo else "otro"])
    print(f"Candidatos: {len(indices)}")
    print(f"Posibles disparos: {total}")
    print(f"Detalle: {salida}")


# def main():
#     parser = argparse.ArgumentParser(description=__doc__)
#     parser.add_argument("entrenamiento", type=Path)
#     parser.add_argument("audio", type=Path)
#     parser.add_argument("--salida", type=Path, default=Path("candidatos.csv"))
#     parser.add_argument("--umbral", type=float, default=0.5)
#     parser.add_argument("--prominencia", type=float, default=0.02)
#     parser.add_argument("--separacion", type=float, default=0.08)
#     args = parser.parse_args()
#     if not 0 <= args.umbral <= 1:
#         parser.error("--umbral debe estar entre 0 y 1")
#     if not 0 < args.prominencia <= 1:
#         parser.error("--prominencia debe estar entre 0 y 1")
#     if not math.isfinite(args.separacion) or args.separacion <= 0:
#         parser.error("--separacion debe ser positiva")
#     modelo = entrenar(args.entrenamiento)
#     contar(modelo, leer_wav(args.audio), args.salida, args.umbral,
#            args.prominencia, args.separacion)


# if __name__ == "__main__":
#     main()

In [10]:
from pathlib import Path

carpeta = Path(r"audios\entrenamiento")
audio = Path(r"audios\Masacre de Ramallo - Test\Audio 1 - Recorte Damian Kuc - Recortado.wav")
salida = Path(r"audios\resultados.csv")

modelo = entrenar(carpeta)
contar(modelo, leer_wav(audio), salida,
       umbral=0.5, prominencia=0.02, separacion=0.08)

no_disparo: 51 recortes
disparo: 167 recortes
Candidatos: 146
Posibles disparos: 3
Detalle: audios\resultados.csv
